실습 3. 긴 누락 구간과 limit
- 연속 결측 길이를 보고 채우기 범위를 제한해 긴 구간 남기기

목표
- 연속 결측 길이를 확인하고 채우기 범위를 제한해 긴 구간을 남기기

단계
- 빈 값이 연속으로 이어진 길이들을 구하기
- 제한 없이 보간하면 모두 채워짐을 확인
- 연속 두 칸까지만 채우면 긴 구간은 남음을 확인

예상 결과
- 연속 결측 길이 10·7·1…, 제한 없으면 0, 두 칸 제한이면 13 남음

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")

# 한글깨짐 해결
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv('../data/22_열처리.csv', encoding='utf-8')

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp').sort_index()
norm = df[['제어출력', '소입로온도']].asfreq('10s')

In [ ]:
# [누락 덩어리(Missing Run-length) 식별 및 보간 한계선 설정]
# 1. `groups = (isna != isna.shift()).cumsum()`: 결측 여부의 변화를 감지해 연속적인 결측 구간들에 유니크한 ID 그룹을 매핑합니다.
# 2. `runs`: 결측 그룹별로 묶어 결측치 개수를 구해 내림차순 정렬합니다. 가장 긴 유실 구간은 10칸(100초) 연속 유실임을 감지합니다.
# 3. `limit=2`: 10칸짜리 연속 결측에 대해 앞의 2칸만 보간으로 채우고 남은 8칸은 NaN 상태를 유지시켜 무리한 추정을 조절합니다.
# * 연속 10칸과 같이 통신이 장기 두절된 대형 사고 구간은 보간법으로 대충 지어내면 위험하므로,
#   임계치(limit)를 넘겨 빈 채로 두고 점검 알람을 울리거나 정비 주기를 잡는 것이 안전한 판단입니다.
# * sort_values()나 sorted() 등으로 결과를 가시적으로 정렬해야 최장 결측 길이를 쉽게 도출할 수 있습니다.

isna = norm['제어출력'].isna()
groups = (isna != isna.shift()).cumsum()
runs = isna.groupby(groups).sum()

# 빈 값이 연속으로 이어진 길이들을 구하기
print('연속 결측:', sorted(runs[runs > 0].tolist(), reverse=True)) # [10, 7, 1, 1, 1, 1, 1]

# 제한 없이 보간하면 모두 채워짐을 확인
print('제한 없음:', int(norm['제어출력'].interpolate().isna().sum())) # 0

# 연속 두 칸까지만 채우면 긴 구간은 남음을 확인
print('두 칸 제한:', int(norm['제어출력'].interpolate(limit=2).isna().sum()))  # 13

연속 결측: [10, 7, 1, 1, 1, 1, 1]
제한 없음: 0
두 칸 제한: 13
